<a href="https://colab.research.google.com/github/Blackthornedejavre/GoogleColap/blob/Rama-editable/Catastro/VecinosProximos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install geopandas
!pip install matplotlib

Debes meter shapefile en la carpeta MyDrive/Vecinosproximos/0.Catastro_Concejo/

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Ruta al archivo .shp
shapefile_path = '/content/drive/MyDrive/Vecinosproximos/0.Catastro_Concejo/QUIROS-ParcelaCatastral.shp'

# Ruta al archivo .dbf
shapefile_path = '/content/drive/MyDrive/Vecinosproximos/0.Catastro_Concejo/QUIROS-ParcelaCatastral.dbf'

# Cargar el shapefile usando Geopandas
gdf = gpd.read_file(shapefile_path)

# Mostrar las primeras filas de la tabla de atributos (.dbf) en formato Markdown
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas

print(gdf.head().to_markdown())

# Visualizar el shapefile
fig, ax = plt.subplots(figsize=(5, 5))
gdf.plot(ax=ax, alpha=0.5, color='gray')  # Parcelas no filtradas en color gris claro
plt.show()


Debes añadir el excel con la columna "Declarar" y debajo las parcelas, depsues guardarlo en la siguiente ruta MyDrive/Vecinosproximos/1.Excel_ParcelasDeclarar/ParcelasDeclarar

In [ ]:
import geopandas as gpd
import pandas as pd

# Ruta al archivo .shp
shapefile_path = '/content/drive/MyDrive/Vecinosproximos/0.Catastro_Concejo/QUIROS-ParcelaCatastral.shp'

# Ruta al archivo Excel con las referencias catastrales
excel_file_path = '/content/drive/MyDrive/Vecinosproximos/1.Excel_ParcelasDeclarar/ParcelasDeclarar.xlsx'

# Cargar el shapefile usando Geopandas
gdf = gpd.read_file(shapefile_path)

# Cargar las referencias catastrales desde el archivo Excel
df_excel = pd.read_excel(excel_file_path, sheet_name='Declarar', engine='openpyxl')

# Filtrar los valores vacíos en la columna ParcelasDeclarar
df_excel.dropna(subset=['Declarar'], inplace=True)

# Convertir las referencias catastrales en una lista
referencias_catastrales = df_excel['Declarar'].tolist()

# Filtrar el shapefile utilizando las referencias catastrales
gdf_filtrado = gdf[gdf['nationalCa'].isin(referencias_catastrales)]

# Mostrar las primeras filas del shapefile filtrado
print(gdf_filtrado.head().to_markdown())

# Graficar las parcelas filtradas en un mapa
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, alpha=0.5, color='gray')  # Parcelas no filtradas en color gris claro
gdf_filtrado.plot(ax=ax, alpha=0.8, color='blue')  # Parcelas filtradas en color azul
plt.title('Parcelas Filtradas')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.show()

Crear un campo de vecinos en la nueva tabla que contiene todos los poligonos proximosa las referencias catastrales que queremos clasificar y los almacena en una nueva columna "vecinos".

In [ ]:

from shapely.geometry import Polygon

# Lista para almacenar los vecinos para cada referencia
vecinos = []

# Función para encontrar polígonos contiguos (vecinos)
def encontrar_vecinos(referencia):
    poligono_referencia = gdf[gdf['nationalCa'] == referencia].geometry.iloc[0]
    vecinos_referencia = []

    for idx, poligono in gdf.iterrows():
        if poligono['nationalCa'] != referencia and poligono.geometry.touches(poligono_referencia):
            vecinos_referencia.append(poligono['nationalCa'])

    return vecinos_referencia

# Recorrer cada referencia catastral y encontrar vecinos
for referencia in referencias_catastrales:
    vecinos_referencia = encontrar_vecinos(referencia)
    vecinos.append({'Referencia Catastral': referencia, 'Vecinos_Proximos': ', '.join(vecinos_referencia)})

# Crear la tabla con los resultados
tabla_resultados_vecinos = pd.DataFrame(vecinos)

# Mostrar la tabla en pantalla
print(tabla_resultados_vecinos.to_markdown())


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Polygon

# Rest of the code to load the shapefile, read the Excel file, and find neighbors (as provided previously)

# Function to extract polygon and parcel numbers from the cadastral reference
def get_polygon_and_parcel(reference):
    polygon = reference[7:10]
    parcel = reference[-5:]
    return polygon, parcel

# Function to plot all neighboring references with black borders and labels
def plot_neighbors(reference, neighbors_gdf):
    fig, ax = plt.subplots(figsize=(10, 10))

    # Plot all neighboring references in blue with black borders
    neighbors_gdf.plot(ax=ax, alpha=0.8, edgecolor='black', linewidth=0.7, facecolor='blue')

    # Plot the cadastral reference in red with black borders
    gdf[gdf['nationalCa'] == reference].plot(ax=ax, alpha=0.8, edgecolor='black', linewidth=0.7, facecolor='red')

    # Add labels for each polygon and parcel from the cadastral reference
    for idx, row in neighbors_gdf.iterrows():
        polygon, parcel = get_polygon_and_parcel(row['nationalCa'])
        label = f'Pol: {polygon}\nParc: {parcel}'
        ax.annotate(label, xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                    xytext=(3, 3), textcoords="offset points", color='black', fontsize=10, ha='center')

    plt.title(f'Referencia Catastral: {reference} y Vecinos Próximos (Azul)')
    plt.xlabel('Longitud')
    plt.ylabel('Latitud')
    plt.show()

# Loop through each row in the table and create plots for cadastral references and their neighbors on the initial shapefile
for _, row in tabla_resultados_vecinos.iterrows():
    referencia_catastral = row['Referencia Catastral']
    vecinos_proximos = row['Vecinos_Proximos'].split(', ')
    neighbors_gdf = gdf[gdf['nationalCa'].isin([referencia_catastral] + vecinos_proximos)]
    plot_neighbors(referencia_catastral, neighbors_gdf)


In [ ]:
import numpy as np
import math
from shapely.geometry import Polygon


# Función para encontrar polígonos contiguos (vecinos) y clasificar sus direcciones
def encontrar_vecinos(referencia):
    poligono_referencia = gdf[gdf['nationalCa'] == referencia].geometry.iloc[0]
    vecinos_referencia = []
    direcciones = []

    for idx, poligono in gdf.iterrows():
        if poligono['nationalCa'] != referencia and poligono.geometry.touches(poligono_referencia):
            vecinos_referencia.append(poligono['nationalCa'])
            centroide_vecino = poligono.geometry.centroid
            dx = centroide_vecino.x - poligono_referencia.centroid.x
            dy = centroide_vecino.y - poligono_referencia.centroid.y
            angulo = np.degrees(math.atan2(dy, dx))

            if angulo < 0:
                angulo += 360

            if angulo >= 315 or angulo < 45:
                direccion = 'Este'
            elif 45 <= angulo < 135:
                direccion = 'Norte'
            elif 135 <= angulo < 225:
                direccion = 'Sur'
            else:
                direccion = 'Oeste'

            direcciones.append(direccion)

    return vecinos_referencia, direcciones

# Recorrer cada referencia catastral y encontrar vecinos y direcciones
for referencia in referencias_catastrales:
    vecinos_referencia, direcciones_referencia = encontrar_vecinos(referencia)
    vecinos.append({'Referencia Catastral': referencia, 'Vecinos': ', '.join([f"{vecino} ({direccion})" for vecino, direccion in zip(vecinos_referencia, direcciones_referencia)])})

# Crear la tabla con los resultados
tabla_resultados_vecinos = pd.DataFrame(vecinos)

# Mostrar la tabla en pantalla
print(tabla_resultados_vecinos.to_markdown())



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import math

# (Tu código para cargar los datos y filtrar las referencias catastrales)

# Definir función para encontrar vecinos y direcciones
def encontrar_vecinos(referencia):
    poligono_referencia = gdf[gdf['nationalCa'] == referencia].geometry.iloc[0]
    vecinos_referencia = []
    direcciones = {'Norte': [], 'Sur': [], 'Este': [], 'Oeste': []}

    for idx, poligono in gdf.iterrows():
        if poligono['nationalCa'] != referencia and poligono.geometry.touches(poligono_referencia):
            vecinos_referencia.append(poligono['nationalCa'])
            centroide_vecino = poligono.geometry.centroid
            dx = centroide_vecino.x - poligono_referencia.centroid.x
            dy = centroide_vecino.y - poligono_referencia.centroid.y
            angulo = np.degrees(math.atan2(dy, dx))

            if angulo < 0:
                angulo += 360

            if angulo >= 315 or angulo < 45:
                direccion = 'Este'
            elif 45 <= angulo < 135:
                direccion = 'Norte'
            elif 135 <= angulo < 225:
                direccion = 'Sur'
            else:
                direccion = 'Oeste'

            direcciones[direccion].append(poligono['nationalCa'])

    return direcciones

# Recorrer cada referencia catastral y encontrar vecinos y direcciones
direcciones_vecinos = []

for referencia in referencias_catastrales:
    direcciones_referencia = encontrar_vecinos(referencia)
    for direccion, vecinos_direccion in direcciones_referencia.items():
        direcciones_vecinos.append({
            'Referencia Catastral': referencia,
            'Dirección': direccion,
            'Vecinos': ', '.join(vecinos_direccion)
        })

# Crear DataFrame con los resultados de vecinos y direcciones
df_direcciones_vecinos = pd.DataFrame(direcciones_vecinos)

# Mostrar tabla en pantalla
print(df_direcciones_vecinos.to_markdown())


In [8]:
# Exportar la tabla a un archivo Excel
excel_output_path = '/content/drive/MyDrive/Vecinosproximos/2.OUT_VecinosProximos/TablaVecinosProx.xlsx'
df_direcciones_vecinos.to_excel(excel_output_path, index=False)
